C[i, j] = Σ_k A[i, k] * B[k, j]     for i in [0,M), j in [0,N)

In [ ]:
def matmul_loops(A, B):
    M, K = len(A), len(A[0])
    K2, N = len(B), len(B[0])
    assert K == K2, "inner dimensions must match"

    C = [[0.0] * N for _ in range(M)]
    for i in range(M):
        for j in range(N):
            s = 0.0
            for k in range(K):
                s += A[i][k] * B[k][j]
            C[i][j] = s
    return C

Complexity: O(M·N·K) multiply-adds. Three nested loops.

Interview follow-up: "Can you make it faster without vectorizing?" → Swap the loop order so the innermost loop is over k with A[i][k] and B[k][j] accessed contiguously. Better: i-k-j order (accumulate C[i][j] += A[i][k] * B[k][j] in the k-loop) — this is cache-friendly because B[k][j] and C[i][j] are both row-major contiguous

In [ ]:
def matmul_ikj(A, B):
    M, K = len(A), len(A[0])
    N = len(B[0])
    C = [[0.0] * N for _ in range(M)]
    for i in range(M):
        for k in range(K):
            a = A[i][k]
            Bk = B[k]
            Ci = C[i]
            for j in range(N):
                Ci[j] += a * Bk[j]     # contiguous access, no repeated indexing
    return C

# this is 3x-5x faster 

Fully Vectorized 

In [ ]:
C = (A[:, :, None] * B[None, :, :]).sum(axis=1)
# shapes: A[:,:,None] → (M, K, 1)
#         B[None,:,:] → (1, K, N)
#         product     → (M, K, N)
#         sum axis=1  → (M, N)

Broadcastign rules 

Broadcasting lets NumPy/PyTorch operate on arrays of different shapes by virtually expanding the smaller one. Two rules:

Rule 1 — Align shapes from the right. Pad the shorter shape with 1s on the left.

Rule 2 — Dimensions are compatible if they're equal, or one of them is 1. A size-1 dimension stretches to match the other.